# Import Packages

In [1]:
import mne
from mne.preprocessing import ICA
import os

RAW_DATA_DIR = "./../data/raw/"
PROCESSED_DATA_DIR = "./../data/processed/"

# Load Data

In [ ]:
raw = mne.io.read_raw_gdf("./../data/raw/A01T.gdf", preload = True, eog = ["EOG-left", "EOG-central", "EOG-right"])
raw.annotations.rename({"769": "left", "770": "right", "771": "foot", "772": "tongue", "1023": "bad_trial"})
raw.plot()
raw

Extracting EDF parameters from /home/jshen/Projects/Summer 2025 Deep Learning/EEG Motor Imagery DL/data/raw/A01T.gdf...
GDF file detected


Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...


/home/jshen/miniconda3/envs/Summer2025DL/lib/python3.13/contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>

# Preprocessing

## Band Pass Filter

In [3]:
# Band Pass Filter (0.5 - 40 Hz)
# 50Hz Notch Filter already applied during data collection
raw.filter(l_freq = 0.5, h_freq = 40.0)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 1651 samples (6.604 s)



<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>

In [4]:
# Verify band pass filter worked
# raw.plot_psd(fmax = 50)

## ICA

In [5]:
n_EEG_channels = sum(1 for ch in raw.info["ch_names"] if ch.startswith("EEG"))
ica = ICA(n_components = n_EEG_channels, random_state = 42)
ica.fit(raw)
ica

Fitting ICA to data using 22 channels (please be patient, this may take a while)
Omitting 28125 of 672528 (4.18%) samples, retaining 644403 (95.82%) samples.
Selecting by number: 22 components
Fitting ICA took 24.1s.


Method,fastica
Fit parameters,algorithm=parallelfun=logcoshfun_args=Nonemax_iter=1000
Fit,49 iterations on raw data (644403 samples)
ICA components,22
Available PCA components,22
Channel types,eeg
ICA components marked for exclusion,—


In [6]:
eog_indices, eog_scores = ica.find_bads_eog(raw)
ica.exclude = eog_indices
ica.apply(raw)

Using EOG channels: EOG-left, EOG-central, EOG-right
Omitting 28125 of 672528 (4.18%) samples, retaining 644403 (95.82%) samples.
Omitting 28125 of 672528 (4.18%) samples, retaining 644403 (95.82%) samples.
Omitting 28125 of 672528 (4.18%) samples, retaining 644403 (95.82%) samples.
Omitting 28125 of 672528 (4.18%) samples, retaining 644403 (95.82%) samples.
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2500 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pas

<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>

In [7]:
# Verify artifacts are removed
# raw.plot()

## Epoch Data by Events

In [8]:
events, event_id = mne.events_from_annotations(raw)
print(events)
print(event_id)

Used Annotations descriptions: [np.str_('1072'), np.str_('276'), np.str_('277'), np.str_('32766'), np.str_('768'), np.str_('foot'), np.str_('left'), np.str_('right'), np.str_('tongue')]
[[     0      0      4]
 [     0      0      2]
 [ 29683      0      4]
 ...
 [668991      0      8]
 [670550      0      5]
 [671050      0      7]]
{np.str_('1072'): 1, np.str_('276'): 2, np.str_('277'): 3, np.str_('32766'): 4, np.str_('768'): 5, np.str_('foot'): 6, np.str_('left'): 7, np.str_('right'): 8, np.str_('tongue'): 9}


In [9]:
event_dict = {
    "left": event_id["left"],
    "right": event_id["right"],
    "foot": event_id["foot"],
    "tongue": event_id["tongue"]
}

epochs = mne.Epochs(
    raw,
    events,
    event_id = event_dict,
    tmin = -0.5,
    tmax = 3.5,
    reject_by_annotation = True,
    preload = True,
    baseline = None,
    verbose = True
)

# Verify epochs
# epochs.plot(n_epochs = 10)

Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 1001 original time points ...


15 bad epochs dropped


In [10]:
epochs.save(os.path.join(PROCESSED_DATA_DIR, "A01T-epochs.fif"), overwrite = True)

Overwriting existing file.
Overwriting existing file.
Overwriting existing file.


/tmp/ipykernel_15037/482604996.py:1: RuntimeWarning: This filename (./../data/processed/A01T-epochs.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs.save(os.path.join(PROCESSED_DATA_DIR, "A01T-epochs.fif"), overwrite = True)


[PosixPath('/home/jshen/Projects/Summer 2025 Deep Learning/EEG Motor Imagery DL/notebooks/../data/processed/A01T-epochs.fif')]